# Experiment B — Random Prompt Generation Baseline

**Evolutionary Prompt Optimization for LLM-Based Sentiment Analysis**  
**Member 4 — Experimental Design and Baseline Implementation**

---

## Purpose

This notebook implements **Experiment B**: the **random prompt generation baseline**.

The key idea is to isolate the contribution of *evolutionary pressure* (selection, crossover, mutation).  
We generate candidate prompts using the **same LLM-based generation mechanism** as the GA's initial population, but we apply **no selection, no crossover, and no mutation**. We simply draw a fixed number of candidates at random and pick the best.

This answers the question:  
> *Is the GA's improvement over manual prompts due to the LLM-guided generation alone, or is evolutionary selection actually necessary?*

| Experiment | Method | Candidates | Evolutionary pressure |
|---|---|---|---|
| A | Manual prompting | 8 (by Member 2) | ✗ |
| **B (this notebook)** | **Random generation** | **36** | **✗** |
| C | Genetic Algorithm | 36 scored (12 × 3 gen) | ✓ |

### Fair-comparison constraints (all three experiments)
- Same LLM, same revision, same greedy decoding
- Same optimisation split, same 1 000-example sample
- Same validation split, same 1 000-example sample
- Same primary metric (Macro F1)
- Comparable evaluation budget: Experiment B scores the same number of
  evaluations as the GA (population × generations = 12 × 3 = 36)
- Test set never used
- Global seed: 42

## 1. Environment Setup

In [ ]:
import os
import sys
import json
import subprocess
import random
import time
import importlib.util
from pathlib import Path

REPO_URL = "https://github.com/sarrazer24/evolutionary-prompt-optimization.git"
REPO_DIR = Path("/kaggle/working/evolutionary-prompt-optimization")

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", "main", "--single-branch", REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    if not (REPO_DIR / ".git").exists():
        raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository.")
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", "main"], check=True)

os.chdir(REPO_DIR)

REQUIRED_PACKAGES = {
    "transformers": "transformers",
    "accelerate": "accelerate",
    "datasets": "datasets",
    "sklearn": "scikit-learn",
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "scipy": "scipy",
}
missing_packages = [
    pkg for mod, pkg in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(mod) is None
]
if missing_packages:
    print("Installing missing packages:", missing_packages)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *missing_packages],
        check=True,
    )
else:
    print("All required packages are already installed.")

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("Project directory:", Path.cwd())
print("Python version:", sys.version)
print("Setup complete.")

## 2. Imports and Reproducibility

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from datetime import datetime
from IPython.display import display

from src.data.sst2_data import load_saved_splits, stratified_sample
from src.llm.config import ModelConfig
from src.llm.model_interface import LLMInterface
from src.prompts.baseline_prompts import get_baseline_prompt_texts
from src.prompts.initial_population import generate_initial_prompts
from src.prompts.validation import normalize_prompt
from src.evaluation.evaluation import PredictionCache, evaluate_prompt, save_results

# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Global seed: {SEED}")

## 3. GPU Check

In [ ]:
if not torch.cuda.is_available():
    print("⚠  No GPU detected — inference will be very slow on CPU.")
    print("   On Kaggle: Settings → Accelerator → GPU T4 x2.")
else:
    for i in range(torch.cuda.device_count()):
        name = torch.cuda.get_device_name(i)
        mem  = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f"GPU {i}: {name}  ({mem:.1f} GB)")

## 4. Experiment Configuration

`N_RANDOM_CANDIDATES` mirrors the GA evaluation budget:
- GA population size: 12 individuals
- GA generations: 3
- Total GA evaluations: 12 × 3 = **36** (excluding re-evaluations of elites)

We generate **36 random candidates** so that Experiments B and C use the same number of LLM calls for prompt scoring.

In [ ]:
# ── Experiment B configuration ────────────────────────────────────────────────

# Must match GA budget (population_size × n_generations from Notebook 05)
GA_POPULATION_SIZE = 12
GA_GENERATIONS     = 3
N_RANDOM_CANDIDATES = GA_POPULATION_SIZE * GA_GENERATIONS  # = 36

N_OPT_EXAMPLES = 1_000
N_VAL_EXAMPLES = 1_000

OUTPUT_DIR = Path("experiments/results/experiment_B_random")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPERIMENT_CONFIG = {
    "experiment": "B",
    "description": "Random prompt generation baseline — no evolutionary pressure",
    "dataset": "nyu-mll/glue:sst2",
    "model_id": "Qwen/Qwen2.5-1.5B-Instruct",
    "inference": {"greedy": True, "max_new_tokens": 8, "batch_size": 32},
    "generation": {
        "method": "llm_assisted_random",
        "n_candidates": N_RANDOM_CANDIDATES,
        "seed_prompts": "8 manual prompts from baseline_prompts.py",
        "no_selection": True,
        "no_crossover": True,
        "no_mutation": True,
    },
    "split_seed": 42,
    "evaluation_seed": SEED,
    "n_opt_examples": N_OPT_EXAMPLES,
    "n_val_examples": N_VAL_EXAMPLES,
    "metric_primary": "macro_f1",
    "metric_secondary": "accuracy",
    "test_set_used": False,
    "created_at": datetime.now().isoformat(),
}

config_path = OUTPUT_DIR / "experiment_config.json"
config_path.write_text(json.dumps(EXPERIMENT_CONFIG, indent=4, ensure_ascii=False))
print(json.dumps(EXPERIMENT_CONFIG, indent=2))

## 5. Load Data

In [ ]:
splits = load_saved_splits()

opt_df = splits.opt
val_df = splits.val

print(f"Optimisation split : {len(opt_df):>6,} rows  "
      f"(positive: {opt_df['label'].mean():.1%})")
print(f"Validation split   : {len(val_df):>6,} rows  "
      f"(positive: {val_df['label'].mean():.1%})")

opt_sample = stratified_sample(
    opt_df,
    n=N_OPT_EXAMPLES,
    seed=SEED
)

val_sample = stratified_sample(
    val_df,
    n=N_VAL_EXAMPLES,
    seed=SEED
)

opt_texts = opt_sample["text"].tolist()
opt_labels = opt_sample["label"].tolist()

val_texts = val_sample["text"].tolist()
val_labels = val_sample["label"].tolist()

print(f"\nOpt sample : {len(opt_texts)} examples  "
      f"(positive: {sum(opt_labels)/len(opt_labels):.1%})")
print(f"Val sample : {len(val_texts)} examples  "
      f"(positive: {sum(val_labels)/len(val_labels):.1%})")

## 6. Load the LLM

In [ ]:
model_config = ModelConfig()
llm = LLMInterface(model_config)
llm.load()
print(f"Model loaded: {llm.model_id}")
print(f"Device      : {llm.device}")

## 7. Random Prompt Generation

### Method

We call `generate_initial_prompts()` — the same function used to seed the GA's generation 0 — but we call it once to produce **all 36 candidates** without any subsequent selection or genetic operators.

**Why use the same generator?**
- It makes the comparison fair: Experiment B uses the same LLM generation quality as the GA.
- Any GA advantage over Experiment B can only come from *evolutionary selection*, not from better individual generation.

Seed prompts: the 8 manual prompts from Member 2 (same seeds used by the GA).

In [ ]:
gen_rng = random.Random(SEED)
seed_prompts = get_baseline_prompt_texts()   # 8 prompts from Member 2

print(f"Generating {N_RANDOM_CANDIDATES} random candidates "
      f"using seed {SEED} and {len(seed_prompts)} seed prompts …")

t0 = time.time()
random_candidates = generate_initial_prompts(
    llm=llm,
    n=N_RANDOM_CANDIDATES,
    seed_prompts=seed_prompts,
    rng=gen_rng,
)
elapsed_gen = time.time() - t0

# Deduplicate while preserving order
seen = set()
unique_candidates = []
for p in random_candidates:
    norm = normalize_prompt(p)
    if norm not in seen:
        seen.add(norm)
        unique_candidates.append(p)

print(f"Generated   : {len(random_candidates)} candidates  ({elapsed_gen:.1f}s)")
print(f"Unique      : {len(unique_candidates)} after deduplication")
print()
for i, p in enumerate(unique_candidates[:5], 1):
    print(f"[{i}] {p[:120]}{'…' if len(p) > 120 else ''}")
if len(unique_candidates) > 5:
    print(f"  … {len(unique_candidates) - 5} more …")

## 8. Evaluate Every Random Candidate on the Optimisation Split

In [ ]:
cache_B = PredictionCache()
opt_results = []

for idx, prompt in enumerate(unique_candidates, 1):
    candidate_id = f"random_{idx:03d}"
    print(f"[{idx}/{len(unique_candidates)}] {candidate_id}")
    t0 = time.time()
    result = evaluate_prompt(
        prompt=prompt,
        examples=opt_texts,
        labels=opt_labels,
        predict_fn=llm.predict_raw_batch,
        model_id=llm.model_id,
        inference_config=llm.inference_config,
        cache=cache_B,
        split_name="opt",
    )
    elapsed = time.time() - t0
    m = result["metrics"]
    opt_results.append({
        "rank": None,
        "candidate_id": candidate_id,
        "prompt_text": prompt,
        "macro_f1": round(m["f1_macro"], 6),
        "accuracy":  round(m["accuracy"],  6),
        "precision_macro": round(m["precision_macro"], 6),
        "recall_macro":    round(m["recall_macro"],    6),
        "n_invalid":  m["n_invalid"],
        "invalid_rate": round(m["invalid_rate"], 6),
        "n_examples": m["n_examples"],
        "n_cached":   result["n_cached"],
        "split": "opt",
        "elapsed_s": round(elapsed, 2),
    })
    print(f"   Macro F1 = {m['f1_macro']:.4f}  |  "
          f"Accuracy = {m['accuracy']:.4f}  |  "
          f"Invalid = {m['n_invalid']}  |  {elapsed:.1f}s")

# Sort and rank
opt_results.sort(key=lambda r: r["macro_f1"], reverse=True)
for rank, r in enumerate(opt_results, 1):
    r["rank"] = rank

print("\nAll random candidates evaluated.")

## 9. Optimisation-Split Leaderboard

In [ ]:
df_opt = pd.DataFrame(opt_results)[
    ["rank", "candidate_id", "macro_f1", "accuracy", "n_invalid", "invalid_rate", "elapsed_s"]
]
display(df_opt.set_index("rank"))

best_candidate = opt_results[0]["candidate_id"]
best_prompt    = opt_results[0]["prompt_text"]
best_f1_opt    = opt_results[0]["macro_f1"]

print(f"\n✔ Best random candidate: '{best_candidate}'  "
      f"(Macro F1 = {best_f1_opt:.4f})")
print(f"  Text: {best_prompt[:200]}")

# Distribution statistics
f1_values = [r["macro_f1"] for r in opt_results]
print(f"\nF1 distribution (all {len(f1_values)} candidates):")
print(f"  Mean   : {np.mean(f1_values):.4f}")
print(f"  Median : {np.median(f1_values):.4f}")
print(f"  Std    : {np.std(f1_values):.4f}")
print(f"  Min    : {np.min(f1_values):.4f}")
print(f"  Max    : {np.max(f1_values):.4f}")

## 10. Visualise Random Baseline Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

f1_sorted  = [r["macro_f1"] for r in opt_results]
acc_sorted = [r["accuracy"] for r in opt_results]
colors     = ["#e74c3c" if i == 0 else "#9b59b6" for i in range(len(f1_sorted))]

# F1 distribution histogram
axes[0].hist(f1_sorted, bins=12, color="#9b59b6", edgecolor="white", alpha=0.85)
axes[0].axvline(best_f1_opt, color="#e74c3c", linewidth=2,
                label=f"Best = {best_f1_opt:.4f}")
axes[0].axvline(np.mean(f1_sorted), color="#f39c12", linewidth=2,
                linestyle="--", label=f"Mean = {np.mean(f1_sorted):.4f}")
axes[0].set_xlabel("Macro F1")
axes[0].set_ylabel("Count")
axes[0].set_title("Experiment B — Random Candidates\nMacro F1 distribution (opt split)")
axes[0].legend()

# Sorted F1 bar chart (top 20)
top_n = min(20, len(opt_results))
top   = opt_results[:top_n]
names = [r["candidate_id"] for r in top][::-1]
f1s   = [r["macro_f1"]    for r in top][::-1]
cols  = ["#e74c3c"] + ["#9b59b6"] * (top_n - 1)
axes[1].barh(names, f1s, color=cols[::-1])
axes[1].set_xlabel("Macro F1")
axes[1].set_title(f"Top {top_n} Random Candidates (opt split)")
axes[1].set_xlim(max(0, min(f1s) - 0.02), min(1, max(f1s) + 0.02))
for v, y in zip(f1s, range(top_n)):
    axes[1].text(v + 0.001, y, f"{v:.4f}", va="center", fontsize=7)

plt.tight_layout()
plot_path = OUTPUT_DIR / "random_candidates_opt.png"
fig.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Figure saved → {plot_path}")

## 11. Validate the Best Random Candidate

Validation is run **exactly once** to report generalisation.
This score feeds into the three-way comparison in Notebook 09.

In [ ]:
print(f"Validating best random candidate: '{best_candidate}'")
print(f"Prompt text: {best_prompt}\n")

t0 = time.time()
val_result = evaluate_prompt(
    prompt=best_prompt,
    examples=val_texts,
    labels=val_labels,
    predict_fn=llm.predict_raw_batch,
    model_id=llm.model_id,
    inference_config=llm.inference_config,
    split_name="val",
)
elapsed_val = time.time() - t0

vm = val_result["metrics"]
val_summary = {
    "candidate_id":    best_candidate,
    "prompt_text":     best_prompt,
    "macro_f1":        round(vm["f1_macro"],       6),
    "accuracy":        round(vm["accuracy"],        6),
    "precision_macro": round(vm["precision_macro"], 6),
    "recall_macro":    round(vm["recall_macro"],    6),
    "n_invalid":       vm["n_invalid"],
    "invalid_rate":    round(vm["invalid_rate"],    6),
    "n_examples":      vm["n_examples"],
    "split":           "val",
    "elapsed_s":       round(elapsed_val, 2),
}

print(f"Validation Macro F1 : {vm['f1_macro']:.4f}")
print(f"Validation Accuracy : {vm['accuracy']:.4f}")
print(f"Invalid rate        : {vm['invalid_rate']:.2%}")
print(f"Elapsed             : {elapsed_val:.1f}s")

## 12. Save Results

In [ ]:
# ── Full opt results ───────────────────────────────────────────────────────────
opt_csv_path  = OUTPUT_DIR / "experiment_B_opt_results.csv"
opt_json_path = OUTPUT_DIR / "experiment_B_opt_results.json"

save_results(opt_results, opt_csv_path,  append=False)
save_results(opt_results, opt_json_path)
print(f"Opt results  → {opt_csv_path}")
print(f"             → {opt_json_path}")

# ── Validation summary ────────────────────────────────────────────────────────
val_csv_path  = OUTPUT_DIR / "experiment_B_val_best.csv"
val_json_path = OUTPUT_DIR / "experiment_B_val_best.json"

save_results(val_summary, val_csv_path,  append=False)
save_results(val_summary, val_json_path)
print(f"Val results  → {val_csv_path}")
print(f"             → {val_json_path}")

# ── Comparison-ready summary (consumed by Notebook 09) ───────────────────────
comparison_payload = {
    "experiment": "B",
    "method": "random_generation",
    "n_candidates": len(unique_candidates),
    "n_opt_examples": N_OPT_EXAMPLES,
    "n_val_examples": N_VAL_EXAMPLES,
    "best_candidate_id": best_candidate,
    "best_prompt_text": best_prompt,
    "best_macro_f1_opt": best_f1_opt,
    "best_accuracy_opt": opt_results[0]["accuracy"],
    "best_macro_f1_val": val_summary["macro_f1"],
    "best_accuracy_val": val_summary["accuracy"],
    "f1_distribution": {
        "mean":   round(float(np.mean(f1_values)),   6),
        "median": round(float(np.median(f1_values)), 6),
        "std":    round(float(np.std(f1_values)),    6),
        "min":    round(float(np.min(f1_values)),    6),
        "max":    round(float(np.max(f1_values)),    6),
    },
    "all_opt_results": opt_results,
    "model_id": llm.model_id,
    "model_revision": model_config.revision,
    "seed": SEED,
    "created_at": datetime.now().isoformat(),
}

summary_path = OUTPUT_DIR / "experiment_B_summary.json"
summary_path.write_text(json.dumps(comparison_payload, indent=4, ensure_ascii=False, default=str))
print(f"Summary      → {summary_path}")

print("\nAll files saved.")

## 13. Final Summary

In [ ]:
print("=" * 60)
print("EXPERIMENT B — RANDOM PROMPT GENERATION BASELINE")
print("=" * 60)
print(f"Candidates generated    : {len(unique_candidates)}")
print(f"Opt examples per prompt : {N_OPT_EXAMPLES}")
print(f"Val examples (best)     : {N_VAL_EXAMPLES}")
print()
print(f"Best candidate          : {best_candidate}")
print(f"  Opt  Macro F1         : {best_f1_opt:.4f}")
print(f"  Val  Macro F1         : {val_summary['macro_f1']:.4f}")
print(f"  Val  Accuracy         : {val_summary['accuracy']:.4f}")
print()
print(f"F1 distribution (all {len(f1_values)} candidates):")
print(f"  Mean = {np.mean(f1_values):.4f}  |  Std = {np.std(f1_values):.4f}  |  "
      f"Min = {np.min(f1_values):.4f}  |  Max = {np.max(f1_values):.4f}")
print()
print("Output files:")
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {f.name}")
print()
print("Next step: run Notebook 09 (Experiment C — Full Three-Way Comparison)")